# Chapitre 2 - Cours 3 : schémas souples et langages de requête

Le cours 1 a pesé la localité, le cours 2 les jointures. Restent deux arguments du débat
entre relationnel et documents : la **souplesse du schéma** et le **langage de requête**.
Ce cours couvre la fin de la section *Relational Versus Document Databases Today*
(*Which data model leads to simpler application code?*, *Schema flexibility in the
document model*, *Convergence of document and relational databases*) et la section
*Query Languages for Data* (*Declarative Queries on the Web*, *MapReduce Querying*).

**Plan.**

1. Quel modèle simplifie le code ?
2. Schéma à la lecture, schéma à l'écriture.
3. Déclaratif contre impératif.
4. Les requêtes déclaratives sur le Web.
5. MapReduce sur des documents.
6. Le pipeline d'agrégation.
7. La convergence des modèles.

In [1]:
import json
import sqlite3
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import datamodels as dm
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 90)
rng = np.random.default_rng(0)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")

## 1. Quel modèle simplifie le code ? (section *Which data model leads to simpler application code?*)

Le verdict du livre tient en deux phrases :

- si les données ont une **structure de document** (un arbre de relations un-à-plusieurs,
  chargé d'un bloc), le modèle document est probablement le bon choix. Le découpage
  relationnel d'un tel arbre en plusieurs tables, le *shredding*, donne des schémas
  lourds et du code inutilement compliqué (cours 1) ;
- si l'application a besoin de relations **plusieurs-à-plusieurs**, le modèle document
  perd son attrait : jointures émulées dans le code, dénormalisation à maintenir
  (cours 2). Et si ces relations dominent, le **graphe** devient le modèle naturel
  (cours 4).

Le modèle document a une autre limite, plus discrète : on ne peut pas désigner
**directement** un élément imbriqué. Il faut dire « le deuxième poste du profil 42 »,
une adresse **positionnelle** qui devient fausse dès que la liste change. En relationnel,
chaque poste a son propre identifiant stable. Démonstration : un autre service a retenu
l'adresse d'un poste pour y attacher une recommandation, puis l'utilisateur supprime un
ancien poste.

In [2]:
doc = {"user_id": 42, "positions": [
    {"job_title": "Architecte de données", "organization": "Datanet"},
    {"job_title": "Ingénieure logiciel", "organization": "Novasoft"},
    {"job_title": "Stagiaire", "organization": "Pixelsoft"}]}
rel = pd.DataFrame({"id": [9001, 9002, 9003], "user_id": 42,
                    "job_title": [p["job_title"] for p in doc["positions"]]})
adresse_doc = ("profil 42", "positions", 1)           # « le 2e poste du profil 42 »
adresse_rel = 9002                                    # l'identifiant du poste

del doc["positions"][0]                               # l'utilisateur supprime son poste le plus récent
rel = rel[rel.id != 9001]
pd.DataFrame({
    "référence": ["positions[1] dans le document", "positions.id = 9002"],
    "désignait": ["Ingénieure logiciel", "Ingénieure logiciel"],
    "désigne maintenant": [doc["positions"][adresse_doc[2]]["job_title"],
                           rel.loc[rel.id == adresse_rel, "job_title"].item()],
})

,référence,désignait,désigne maintenant
0,positions[1] dans le document,Ingénieure logiciel,Stagiaire
1,positions.id = 9002,Ingénieure logiciel,Ingénieure logiciel


La référence positionnelle pointe désormais sur le **mauvais** poste, sans aucune erreur.
Le remède est d'attribuer un identifiant à chaque élément imbriqué, ce qui revient à
réintroduire une idée relationnelle dans le document.

> **À retenir.** Arbre lu d'un bloc : modèle document. Relations plusieurs-à-plusieurs :
> relationnel, ou graphe si elles dominent. Un élément imbriqué dans un document n'a pas
> d'adresse stable sans identifiant propre.